# PSD — Klasifikasi Sawah/Non-Sawah dengan Random Forest

Alur: sample manual tervalidasi → ekstraksi B02/B03/B04/B08 → evaluasi Random Forest → klasifikasi seluruh raster Sentinel-2A → peta.

## 1. Ekstraksi fitur dari sample manual

Sample harus memiliki field `label` dengan nilai `sawah` atau `non_sawah` dan berjumlah tepat 50 per kelas. Label tidak dibentuk dari NDVI/SCL.

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask


BASE = Path(__file__).resolve().parent

# === UBAH BAGIAN INI SESUAI FILE SAMPLE MANUAL KAMU ===
AOI_PATH = BASE / "data" / "geojson" / "menganti_aoi.geojson"
SAMPLE_PATH = BASE / "data" / "training" / "manual" / "samples.shp"
RASTER_PATH = BASE / "data" / "sentinel2" / "menganti_sentinel2_2026-10-01.tif"

OUTPUT_PATH = (
    BASE / "data" / "training" / "sentinel2_samples_manual.csv"
)

# Nama field label di shapefile.
# Isi atributnya harus: sawah / non_sawah
LABEL_FIELD = "label"

EXPECTED_PER_CLASS = 50

# Band Sentinel-2 pada GeoTIFF:
# 1=B02, 2=B03, 3=B04, 4=B08, 5=SCL, 6=dataMask
BAND_INFO = {
    "B02_mean": 1,
    "B03_mean": 2,
    "B04_mean": 3,
    "B08_mean": 4,
}


def main():
    print("=" * 70)
    print("EKSTRAKSI FITUR SAMPLE MANUAL")
    print("=" * 70)

    if not AOI_PATH.exists():
        raise FileNotFoundError(f"AOI tidak ditemukan: {AOI_PATH}")

    if not SAMPLE_PATH.exists():
        raise FileNotFoundError(
            f"Sample manual tidak ditemukan: {SAMPLE_PATH}\n"
            "Buat polygon sample di QGIS dan isi field 'label' "
            "dengan sawah/non_sawah."
        )

    if not RASTER_PATH.exists():
        raise FileNotFoundError(f"Raster tidak ditemukan: {RASTER_PATH}")

    samples = gpd.read_file(SAMPLE_PATH)

    if LABEL_FIELD not in samples.columns:
        raise ValueError(
            f"Field '{LABEL_FIELD}' tidak ditemukan.\n"
            f"Field tersedia: {list(samples.columns)}"
        )

    samples[LABEL_FIELD] = (
        samples[LABEL_FIELD]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    allowed = {"sawah", "non_sawah"}
    invalid = sorted(set(samples[LABEL_FIELD]) - allowed)

    if invalid:
        raise ValueError(
            f"Label tidak valid: {invalid}. "
            f"Gunakan hanya 'sawah' atau 'non_sawah'."
        )

    print("\nJumlah sample:")
    print(samples[LABEL_FIELD].value_counts())

    for label in ["sawah", "non_sawah"]:
        n = int((samples[LABEL_FIELD] == label).sum())
        if n != EXPECTED_PER_CLASS:
            raise ValueError(
                f"Jumlah {label} = {n}, "
                f"harus tepat {EXPECTED_PER_CLASS}."
            )

    aoi = gpd.read_file(AOI_PATH)

    with rasterio.open(RASTER_PATH) as src:
        raster_crs = src.crs

        if samples.crs is None:
            raise ValueError("Sample tidak memiliki CRS.")

        if aoi.crs is None:
            raise ValueError("AOI tidak memiliki CRS.")

        samples = samples.to_crs(raster_crs)
        aoi = aoi.to_crs(raster_crs)

        # Cek sample berada di dalam/beririsan dengan AOI
        aoi_geom = aoi.geometry.union_all()
        outside = ~samples.geometry.intersects(aoi_geom)

        if outside.any():
            bad = samples.index[outside].tolist()
            raise ValueError(
                f"Ada sample di luar AOI: index {bad}"
            )

        rows = []

        for idx, row in samples.iterrows():
            geom = [row.geometry]

            values, _ = mask(
                src,
                geom,
                crop=True,
                filled=False
            )

            # values shape = (6, h, w)
            valid = ~values.mask[0]

            if not valid.any():
                raise ValueError(
                    f"Sample index {idx} tidak memiliki pixel raster valid."
                )

            result = {
                "sample_id": idx + 1,
                "label": row[LABEL_FIELD],
                "valid_pct": float(valid.mean() * 100.0),
            }

            for feature_name, band_number in BAND_INFO.items():
                band = values[band_number - 1]

                band_values = band.compressed().astype(np.float32)

                if len(band_values) == 0:
                    result[feature_name] = np.nan
                else:
                    result[feature_name] = float(
                        np.mean(band_values)
                    )

            rows.append(result)

    df = pd.DataFrame(rows)

    feature_cols = list(BAND_INFO.keys())

    if df[feature_cols].isna().any().any():
        bad = df[df[feature_cols].isna().any(axis=1)]
        raise ValueError(
            f"Ada sample dengan fitur kosong:\n{bad}"
        )

    # Statistik akhir
    print("\nJumlah sample final:")
    print(df["label"].value_counts())

    print("\nPreview:")
    print(df.head())

    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    df.to_csv(
        OUTPUT_PATH,
        index=False
    )

    print("\nCSV berhasil dibuat:")
    print(OUTPUT_PATH)


if __name__ == "__main__":
    main()


## 2. Pelatihan dan evaluasi Random Forest

Model dievaluasi dengan hold-out 70:30 dan 5-fold cross-validation, kemudian dilatih ulang menggunakan seluruh 100 sample.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


BASE = Path(__file__).resolve().parent

CSV_PATH = (
    BASE / "data" / "training"
    / "sentinel2_samples_manual.csv"
)

MODEL_PATH = (
    BASE / "data" / "output"
    / "random_forest_sawah.joblib"
)

RESULT_PATH = (
    BASE / "data" / "output"
    / "random_forest_evaluation.csv"
)

FEATURES = [
    "B02_mean",
    "B03_mean",
    "B04_mean",
    "B08_mean",
]

TARGET = "label"


def main():
    print("=" * 70)
    print("RANDOM FOREST — KLASIFIKASI SAWAH / NON-SAWAH")
    print("=" * 70)

    df = pd.read_csv(CSV_PATH)

    print("\nData:", df.shape)
    print("\nDistribusi label:")
    print(df[TARGET].value_counts())

    X = df[FEATURES].astype(np.float32)
    y = df[TARGET].astype(str)

    # Hold-out 70:30
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        random_state=42,
        stratify=y,
    )

    model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    print("\n" + "=" * 70)
    print("EVALUASI HOLD-OUT 70:30")
    print("=" * 70)

    print(f"\nAccuracy: {accuracy:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            digits=4
        )
    )

    print("Confusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            y_pred,
            labels=["non_sawah", "sawah"]
        )
    )

    # 5-fold cross validation
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    cv_scores = cross_val_score(
        model,
        X,
        y,
        cv=cv,
        scoring="accuracy"
    )

    print("\n5-Fold Cross Validation:")
    print("Scores :", cv_scores)
    print(f"Mean   : {cv_scores.mean():.4f}")
    print(f"Std    : {cv_scores.std():.4f}")

    # Feature importance
    importance = pd.DataFrame({
        "feature": FEATURES,
        "importance": model.feature_importances_,
    }).sort_values(
        "importance",
        ascending=False
    )

    print("\nFeature Importance:")
    print(importance.to_string(index=False))

    # Latih ulang model final dengan seluruh data manual 100 sample
    final_model = RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1,
    )

    final_model.fit(X, y)

    MODEL_PATH.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    joblib.dump(
        {
            "model": final_model,
            "features": FEATURES,
            "classes": ["non_sawah", "sawah"],
        },
        MODEL_PATH,
    )

    eval_df = pd.DataFrame({
        "metric": [
            "holdout_accuracy",
            "cv_mean_accuracy",
            "cv_std_accuracy",
        ],
        "value": [
            accuracy,
            cv_scores.mean(),
            cv_scores.std(),
        ]
    })

    eval_df.to_csv(
        RESULT_PATH,
        index=False
    )

    print("\nModel final disimpan:")
    print(MODEL_PATH)

    print("\nEvaluasi disimpan:")
    print(RESULT_PATH)


if __name__ == "__main__":
    main()


## 3. Klasifikasi seluruh citra Sentinel-2A

Kode menghasilkan GeoTIFF: `0=non_sawah`, `1=sawah`, `255=NoData`.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import rasterio


BASE = Path(__file__).resolve().parent

MODEL_PATH = (
    BASE / "data" / "output"
    / "random_forest_sawah.joblib"
)

RASTER_PATH = (
    BASE / "data" / "sentinel2"
    / "menganti_sentinel2_2026-10-01.tif"
)

OUTPUT_PATH = (
    BASE / "data" / "output"
    / "klasifikasi_sawah_random_forest.tif"
)

BATCH_SIZE = 100_000


def main():
    print("=" * 70)
    print("KLASIFIKASI SELURUH RASTER — RANDOM FOREST")
    print("=" * 70)

    bundle = joblib.load(MODEL_PATH)

    model = bundle["model"]
    features = bundle["features"]

    print("Features:", features)

    with rasterio.open(RASTER_PATH) as src:
        print("\nRaster:")
        print("Width :", src.width)
        print("Height:", src.height)
        print("Bands :", src.count)
        print("CRS   :", src.crs)
        print("Res   :", src.res)

        # 1=B02, 2=B03, 3=B04, 4=B08
        b02 = src.read(1).astype(np.float32)
        b03 = src.read(2).astype(np.float32)
        b04 = src.read(3).astype(np.float32)
        b08 = src.read(4).astype(np.float32)

        # Band 6 = dataMask
        data_mask = src.read(6)

        pixels = np.column_stack([
            b02.ravel(),
            b03.ravel(),
            b04.ravel(),
            b08.ravel(),
        ])

        valid = (
            (data_mask.ravel() > 0)
            & np.all(np.isfinite(pixels), axis=1)
        )

        valid_indices = np.flatnonzero(valid)

        print("\nTotal pixel :", f"{len(pixels):,}")
        print("Valid pixel :", f"{len(valid_indices):,}")

        prediction = np.full(
            len(pixels),
            255,
            dtype=np.uint8
        )

        label_to_code = {
            "non_sawah": 0,
            "sawah": 1,
        }

        for start in range(
            0,
            len(valid_indices),
            BATCH_SIZE
        ):
            end = min(
                start + BATCH_SIZE,
                len(valid_indices)
            )

            indices = valid_indices[start:end]

            batch = pixels[indices]

            labels = model.predict(batch)

            prediction[indices] = np.array(
                [
                    label_to_code[label]
                    for label in labels
                ],
                dtype=np.uint8
            )

            print(
                f"Progress: {end:,}/{len(valid_indices):,}"
            )

        classified = prediction.reshape(
            src.height,
            src.width
        )

        profile = src.profile.copy()

        profile.update(
            count=1,
            dtype="uint8",
            nodata=255,
            compress="lzw",
        )

        OUTPUT_PATH.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        with rasterio.open(
            OUTPUT_PATH,
            "w",
            **profile
        ) as dst:
            dst.write(classified, 1)
            dst.set_band_description(
                1,
                "Klasifikasi Sawah/Non-Sawah"
            )

            dst.update_tags(
                1,
                CLASS_0="non_sawah",
                CLASS_1="sawah",
                NODATA="255",
                MODEL="Random Forest",
                N_ESTIMATORS="300",
            )

        # Statistik
        valid_count = np.sum(
            np.isin(classified, [0, 1])
        )

        rows = []

        pixel_area_m2 = (
            abs(src.transform.a)
            * abs(src.transform.e)
        )

        for code, label in [
            (0, "non_sawah"),
            (1, "sawah")
        ]:
            n = int(
                np.sum(classified == code)
            )

            area_m2 = n * pixel_area_m2

            rows.append({
                "label": label,
                "pixel": n,
                "area_m2": area_m2,
                "area_km2": area_m2 / 1_000_000,
                "percent_valid": (
                    n / valid_count * 100
                ),
            })

        stats = pd.DataFrame(rows)

        print("\n" + "=" * 70)
        print("HASIL KLASIFIKASI")
        print("=" * 70)
        print(
            stats.to_string(index=False)
        )

    print("\nOutput raster:")
    print(OUTPUT_PATH)


if __name__ == "__main__":
    main()


## 4. Visualisasi hasil klasifikasi

Raster hasil ditampilkan langsung menggunakan Python.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch


BASE = Path(__file__).resolve().parent

RASTER_PATH = (
    BASE / "data" / "output"
    / "klasifikasi_sawah_random_forest.tif"
)

with rasterio.open(RASTER_PATH) as src:
    classified = src.read(1)
    transform = src.transform

plot_data = classified.astype(float)
plot_data[plot_data == 255] = np.nan

cmap = ListedColormap([
    "#d9d9d9",  # non-sawah
    "#18c95b",  # sawah
])

fig, ax = plt.subplots(
    figsize=(12, 8)
)

ax.imshow(
    plot_data,
    cmap=cmap,
    vmin=0,
    vmax=1
)

ax.set_title(
    "Klasifikasi Lahan Sawah dan Non-Sawah\n"
    "Sentinel-2A | Random Forest"
)

ax.axis("off")

legend = [
    Patch(
        facecolor="#d9d9d9",
        edgecolor="black",
        label="Non-Sawah"
    ),
    Patch(
        facecolor="#18c95b",
        edgecolor="black",
        label="Sawah"
    ),
]

ax.legend(
    handles=legend,
    loc="lower right"
)

plt.tight_layout()
plt.show()
